# 08 Qwen2.5-Coder-0.5B: LoRA dengan protokol CodeT5
Konfigurasi LoRA yang sama, tetapi input dan seleksi disamakan dengan CodeT5-small: sumber 400 token (question + code, lalu "\nanswer:"),
target 30 token, seleksi dengan seluruh 7.000 dev beam 10, test beam 10, checkpoint tiap epoch disimpan.
Ini run acuan Qwen di laporan (runs/ft_qwen_python_lora_proto, test BERTScore F1 89,19 mentah / 35,95 rescaled).

## Setup
Pakai kernel Python dari ~/venvs/dq (venv yang menjalankan semua skrip). Sel ini menyamakan lingkungan dengan bg.sh:
cwd di codeqa_exp/code, pylib_metrics di path, Java untuk METEOR, dan GPU 0.

In [ ]:
import os, sys, json
from pathlib import Path

HOME = Path.home()
EXP = HOME / "inahuy/kacuy/codeqa_exp"
CODE = EXP / "code"
os.chdir(CODE)  # scripts import kc_common / registry from here
for p in (str(CODE), str(EXP / "pylib_metrics")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{EXP / 'pylib_metrics'}:{os.environ.get('PYTHONPATH', '')}"
os.environ["PATH"] = f"{HOME / 'envs/kc_tools/bin'}:{os.environ['PATH']}"  # Java for METEOR
os.environ["JAVA_HOME"] = str(HOME / "envs/kc_tools")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("cwd:", os.getcwd(), "| python:", sys.executable)

## ft_qwen_lora_proto.py

In [ ]:
# Default 'smoke' (beberapa langkah saja) supaya Run All tidak langsung memulai training berjam-jam.
VARIANTS = {
    'smoke': ['--run-tag', 'ft_lora_proto_smoke', '--epochs', '1', '--max-steps', '10', '--dev-n', '32', '--test-n', '32'],
    'lora_proto': ['--run-tag', 'ft_qwen_python_lora_proto', '--epochs', '5'],
}
RUN = 'smoke'  # ganti ke salah satu kunci VARIANTS
ARGS = VARIANTS[RUN]
sys.argv = ['ft_qwen_lora_proto.py', *ARGS]
print(RUN, ARGS)

**Sumber:** code/ft_qwen_lora_proto.py

In [ ]:
import argparse, json, math, random
import torch
import bert_score
import kc_common as kc
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from peft import LoraConfig, PeftModel, get_peft_model

In [ ]:
# LoRA on Qwen2.5-Coder-0.5B, set up to match the CodeT5 protocol: 400-token source, 30-token target,
# effective batch 64, grad clip 1.0, linear schedule with 10% warmup, weight decay 0,
# checkpoint chosen by BERTScore F1 on the FULL dev set (beam 10), test with beam 10.
p = argparse.ArgumentParser()
p.add_argument("--model", default="Qwen/Qwen2.5-Coder-0.5B")
p.add_argument("--run-tag", required=True)
p.add_argument("--epochs", type=int, default=5)
p.add_argument("--lr", type=float, default=2e-4)
p.add_argument("--micro", type=int, default=8)
p.add_argument("--accum", type=int, default=8)
p.add_argument("--lora-r", type=int, default=16)
p.add_argument("--max-source", type=int, default=400)
p.add_argument("--max-new", type=int, default=30)
p.add_argument("--dev-n", type=int, default=0)
p.add_argument("--test-n", type=int, default=0)
p.add_argument("--dev-beams", type=int, default=10)
p.add_argument("--max-steps", type=int, default=0)
args = p.parse_args(ARGS)
torch.manual_seed(42)

out = kc.EXP / "runs" / args.run_tag
out.mkdir(parents=True, exist_ok=True)

In [ ]:
def load(split):
    d = kc.DATA / split
    C = (d / f"{split}.code").read_text(encoding="utf-8").splitlines()
    Q = (d / f"{split}.question").read_text(encoding="utf-8").splitlines()
    A = (d / f"{split}.answer").read_text(encoding="utf-8").splitlines()
    assert len(C) == len(Q) == len(A), split
    return [{"id": i, "code": c.strip(), "question": q.strip(), "answer": a.strip()} for i, (c, q, a) in enumerate(zip(C, Q, A))]

In [ ]:
tok = AutoTokenizer.from_pretrained(args.model)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(args.model).cuda()
lcfg = LoraConfig(r=args.lora_r, lora_alpha=2 * args.lora_r, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lcfg)
model.print_trainable_parameters()

SUFFIX = tok("\nanswer:", add_special_tokens=False)["input_ids"]

In [ ]:
def src_ids(r):
    head = tok(f"question: {r['question']}\ncode: {r['code']}", add_special_tokens=False)["input_ids"]
    return head[: args.max_source - len(SUFFIX)] + SUFFIX

def encode(r):
    s = src_ids(r)
    a_ids = tok(" " + r["answer"], add_special_tokens=False)["input_ids"][: args.max_new - 1] + [tok.eos_token_id]
    return s + a_ids, [-100] * len(s) + a_ids

def collate(items):
    L = max(len(x) for x, _ in items)
    ids = torch.full((len(items), L), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(items), L), -100, dtype=torch.long)
    att = torch.zeros((len(items), L), dtype=torch.long)
    for j, (x, y) in enumerate(items):
        ids[j, : len(x)] = torch.tensor(x); lab[j, : len(y)] = torch.tensor(y); att[j, : len(x)] = 1
    return ids, att, lab

In [ ]:
@torch.no_grad()
def generate(rows, beams, batch=16):
    model.eval()
    enc = [src_ids(r) for r in rows]
    order = sorted(range(len(enc)), key=lambda i: -len(enc[i]))
    preds = [""] * len(enc)
    for s in range(0, len(order), batch):
        idx = order[s : s + batch]
        W = max(len(enc[i]) for i in idx)
        ids = torch.full((len(idx), W), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros_like(ids)
        for j, i in enumerate(idx):
            n = len(enc[i]); ids[j, W - n :] = torch.tensor(enc[i]); att[j, W - n :] = 1
        with torch.autocast("cuda", dtype=torch.bfloat16):
            gen = model.generate(input_ids=ids.cuda(), attention_mask=att.cuda(), num_beams=beams,
                                 max_new_tokens=args.max_new, do_sample=False,
                                 pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        for i, t in zip(idx, tok.batch_decode(gen[:, W:], skip_special_tokens=True)):
            preds[i] = " ".join(t.split()).replace("|||", " ")
    model.train()
    return preds

In [ ]:
train = [encode(r) for r in load("train")]
dev = load("dev")[: args.dev_n] if args.dev_n else load("dev")
scorer = bert_score.BERTScorer(model_type="roberta-large", lang="en", device="cuda")
config = {"experiment": "FT_LORA_CODET5PROTO", "model": args.model, "mode": "full", "epochs": args.epochs, "lr": args.lr,
          "lora_r": args.lora_r, "lora_alpha": 2 * args.lora_r, "lora_dropout": 0.05,
          "lora_targets": "q,k,v,o,gate,up,down", "micro_batch": args.micro, "effective_batch": args.micro * args.accum,
          "scheduler": "linear_warmup_0.1", "weight_decay": 0.0, "grad_clip": 1.0, "max_source": args.max_source,
          "max_new_tokens": args.max_new, "dev_selection": f"BERTScore F1, {len(dev)} dev rows, beam {args.dev_beams}",
          "test_decoding": "beam10"}
run = kc.wandb_init(out, args.run_tag, "FT", ["qwen25coder05b", "full", "lora", "codet5proto"], config, "train")

opt = torch.optim.AdamW([q for q in model.parameters() if q.requires_grad], lr=args.lr, weight_decay=0.0)
steps_per_epoch = math.ceil(len(train) / (args.micro * args.accum))
total = args.max_steps or steps_per_epoch * args.epochs
sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
rng = random.Random(42)
best, best_ep, step, history = -1.0, -1, 0, []
model.train()

In [ ]:
for ep in range(args.epochs):
    idx = list(range(len(train))); rng.shuffle(idx)
    batches = [idx[s : s + args.micro] for s in range(0, len(idx), args.micro)]
    running = []
    for bi, b in enumerate(batches):
        ids, att, lab = collate([train[i] for i in b])
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = model(input_ids=ids.cuda(), attention_mask=att.cuda(), labels=lab.cuda()).loss
        (loss / args.accum).backward()
        running.append(loss.item())
        if (bi + 1) % args.accum == 0 or bi == len(batches) - 1:
            torch.nn.utils.clip_grad_norm_([q for q in model.parameters() if q.requires_grad], 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if step % 100 == 0:
                print(f"ep {ep} step {step}/{total} loss {sum(running[-200:]) / len(running[-200:]):.4f}", flush=True)
            if args.max_steps and step >= args.max_steps:
                break
    dp = generate(dev, beams=args.dev_beams)
    f1 = scorer.score(dp, [r["answer"] for r in dev])[2].mean().item() * 100
    history.append({"epoch": ep, "train_loss": sum(running) / len(running), "dev_BERTScore_F1": f1, "steps": step})
    print("EPOCH", json.dumps(history[-1]), flush=True)
    run.log({"dev_BERTScore_F1": f1, "train_loss": history[-1]["train_loss"], "epoch": ep})
    model.save_pretrained(out / f"epoch{ep}")
    if f1 > best:
        best, best_ep = f1, ep
        model.save_pretrained(out / "best_bertscore"); tok.save_pretrained(out / "best_bertscore")
    if args.max_steps and step >= args.max_steps:
        break

base_model = AutoModelForCausalLM.from_pretrained(args.model).cuda()
model = PeftModel.from_pretrained(base_model, out / "best_bertscore").cuda()
test = load("test")[: args.test_n] if args.test_n else load("test")
preds = generate(test, beams=10)
refs = [r["answer"] for r in test]
metrics, per = kc.evaluate(preds, refs)

In [ ]:
P, R, F = scorer.score(preds, refs)
metrics.update({"BERTScore_P": P.mean().item() * 100, "BERTScore_R": R.mean().item() * 100, "BERTScore_F1": F.mean().item() * 100})
config.update({"selected_epoch": best_ep, "best_dev_BERTScore_F1": best, "n_test": len(test)})
kc.write_json(out / "results.json", {"run_tag": args.run_tag, **config, "history": history, "test": metrics})
kc.write_predictions(out / "test_predictions.jsonl", test, preds, per, "FULL", [r["id"] for r in test])
records = kc.read_jsonl(out / "test_predictions.jsonl")
kc.wandb_finalize(run, out, args.run_tag, metrics, records)
run.finish()
print("DONE", args.run_tag, json.dumps({k: round(metrics[k], 3) for k in ("BERTScore_F1", "BLEU", "F1", "EM")}), flush=True)

In [ ]:
# Opsional untuk run panjang: jalankan varian terpilih di tmux lewat bg.sh (log di logs/<RUN>.log),
# supaya training tidak mati kalau browser/kernel putus. Hapus tanda # untuk memakai.
# import subprocess
# subprocess.run([str(EXP / "bg.sh"), RUN, sys.executable, 'ft_qwen_lora_proto.py', *ARGS], check=True)

### Ringkasan hasil test (dibaca dari runs/<tag>/results.json)

In [ ]:
TAGS = ['ft_lora_proto_smoke', 'ft_qwen_python_lora_proto']
for tag in TAGS:
    for cand in (tag, "smoke_" + tag):
        f = EXP / "runs" / cand / "results.json"
        if f.exists():
            r = json.loads(f.read_text())
            t = r.get("test", {})
            print(cand, {k: round(v, 2) for k, v in t.items() if isinstance(v, (int, float))})
            break
    else:
        print(tag, "-> belum ada results.json")